In [37]:
import requests
import pandas as pd
import math
import re
from bs4 import BeautifulSoup

# for plotting
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

## Collection

In [38]:
def get_user_diary(username):
    """Creates a DataFrame of a user's Letterboxd diary given their username

    Args: 
        username (string): Letterboxd username

    Returns:
        df (DataFrame): DataFrame containing the information in a user's diary (movies, ratings, etc.)
    """

    # Finding the max page number
    url = f'https://letterboxd.com/{username}/films/diary/'
    html = requests.get(url).text
    soup = BeautifulSoup(html, 'html.parser')
    pagination = soup.find('div', class_='pagination')
    if pagination:
        max_page = int(pagination.find_all('li', class_='paginate-page')[-1].text) 
    else:
        max_page = 1

    all_data = [] 
    current_month = None 

    for i in range(1, max_page + 1):
        url = f'https://letterboxd.com/{username}/films/diary/page/{i}'
        html = requests.get(url).text
        soup = BeautifulSoup(html, 'html.parser')

        month_watched = soup.find_all(class_='td-calendar')
        day_watched = soup.find_all(class_='td-day diary-day center')
        films = soup.find_all('h3', class_='headline-3 prettify')
        released_dates = soup.find_all(class_='td-released center')
        ratings = soup.find_all('span', class_='rating')

        for j in range(len(films)): 
            if month_watched:
                current_month = month_watched[j].get_text(strip=True) if month_watched[j].get_text(strip=True) else current_month # Update if new month found, else keep previous
            
            data = {
                'Month': current_month,
                'Day': day_watched[j].get_text(strip=True) if j < len(day_watched) else None,
                'Film': films[j].get_text(strip=True),
                'Released': released_dates[j].get_text(strip=True) if j < len(released_dates) else None,
                'Ratings': ratings[j].get_text(strip=True) if j < len(ratings) else None
            }
            all_data.append(data)

    df = pd.DataFrame(all_data)
    return df

df = get_user_diary('khff')
print(df) 

        Month Day                             Film Released Ratings
0     Oct2024  23                Woman of the Hour     2023     ★★★
1     Oct2024  22                             MadS     2024    ★★★★
2     Oct2024  21                       Idle Hands     1999     ★★★
3     Oct2024  21                      Hollow Gate     1988      ★★
4     Oct2024  20                       Wrong Turn     2003     ★★★
...       ...  ..                              ...      ...     ...
3206  Sep1997  27                       Wishmaster     1997        
3207  Dec1996  28  Beavis and Butt-Head Do America     1996     ★★★
3208  Dec1996  27  Beavis and Butt-Head Do America     1996     ★★★
3209  Dec1996  26  Beavis and Butt-Head Do America     1996     ★★★
3210  Dec1996  24  Beavis and Butt-Head Do America     1996     ★★★

[3211 rows x 5 columns]


## Cleaning

In [39]:
df.head(10)

,Month,Day,Film,Released,Ratings
0,Oct2024,23,Woman of the Hour,2023,★★★
1,Oct2024,22,MadS,2024,★★★★
2,Oct2024,21,Idle Hands,1999,★★★
3,Oct2024,21,Hollow Gate,1988,★★
4,Oct2024,20,Wrong Turn,2003,★★★
5,Oct2024,19,RATS!,2024,
6,Oct2024,16,Puppet Master II,1990,★★
7,Oct2024,15,Night of the Scarecrow,1995,★★★
8,Oct2024,13,Daddy's Head,2024,★★★½
9,Oct2024,11,Grave Encounters,2011,


In [40]:
def clean_diary_data(df):
    """Cleans the DataFrame containing Letterboxd diary data.

    Args:
        df (DataFrame): DataFrame from get_user_diary function.

    Returns:
        df (DataFrame): Cleaned DataFrame.
    """

    # extracting month and year into separate columns
    df[['Month', 'Year']] = df['Month'].str.extract('([a-zA-Z]+)(\d{4})')
    month_mapping = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    df['Month'] = df['Month'].map(month_mapping)
    df['Year'] = df['Year'].astype(int)

    # Convert star ratings to numerical values (handle NaN)
    def convert_rating(rating_str):
        if not rating_str:
            return 0  # Replace NaN with 0
        full_stars = rating_str.count('★')
        half_star = 0.5 if '½' in rating_str else 0
        return full_stars + half_star

    df['Ratings'] = df['Ratings'].apply(convert_rating)

    return df

In [41]:
def remove_duplicate_movies(df):
    """Removes duplicate movie entries from the DataFrame.

    Args:
        df (DataFrame): DataFrame with movie entries.

    Returns:
        df (DataFrame): DataFrame with duplicate movie entries removed. 
    """
    # Remove duplicates based on 'Film', 'Year', 'Month', and 'Day'
    df = df.drop_duplicates(subset=['Film', 'Year', 'Month', 'Day'], keep='first') 
    return df

In [42]:
df = clean_diary_data(df)

# uncomment below for removing duplicate movies
# df = remove_duplicate_movies(df)

df.head(10)

,Month,Day,Film,Released,Ratings,Year
0,10,23,Woman of the Hour,2023,3.0,2024
1,10,22,MadS,2024,4.0,2024
2,10,21,Idle Hands,1999,3.0,2024
3,10,21,Hollow Gate,1988,2.0,2024
4,10,20,Wrong Turn,2003,3.0,2024
5,10,19,RATS!,2024,0.0,2024
6,10,16,Puppet Master II,1990,2.0,2024
7,10,15,Night of the Scarecrow,1995,3.0,2024
8,10,13,Daddy's Head,2024,3.5,2024
9,10,11,Grave Encounters,2011,0.0,2024


## Plotting

## EDA